# 5. From handwritten digits to real EEG
**Now change the scientific contract, not just the input shape.** Decode left- versus right-hand motor imagery using public PhysioNet EEG Motor Movement/Imagery v1.0.0 recordings.

The provided cells download runs 4, 8, 12 from participants 1–9. Train people: 1–5; validation people: 6–7; test people: 8–9. Targets vary within each person, so evaluate task epochs and summarize each test person's accuracy; never average all task probabilities into one diagnosis per person. This small demonstration is not clinical or population-level validation. Downloads require internet and may take time; data remain cached for later runs.

## 1. Fixed teaching contract


In [ ]:
ARCH = 'public-eeg-cnn'
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 32
MODEL_SEED = 17
SPLIT_SEED = 17
USE_GPU = False
SUBJECTS = list(range(1, 10))
RUNS = [4, 8, 12]
CHANNELS = ['C3', 'Cz', 'C4']
TRAIN_PEOPLE, VAL_PEOPLE, TEST_PEOPLE = {1,2,3,4,5}, {6,7}, {8,9}
assert TRAIN_PEOPLE.isdisjoint(VAL_PEOPLE | TEST_PEOPLE)
assert VAL_PEOPLE.isdisjoint(TEST_PEOPLE)


## 2. Prepare tools
Jupyter uses the same cells as Colab. No private storage or credentials are required for these public recordings.

In [ ]:
import os, sys, subprocess, importlib.util, json, hashlib, platform
from pathlib import Path
from uuid import uuid4
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
packages = {'tensorflow': 'tensorflow>=2.16,<3', 'numpy': 'numpy>=1.26,<3',
            'matplotlib': 'matplotlib>=3.7,<4', 'pandas': 'pandas>=2,<3',
            'sklearn': 'scikit-learn>=1.4,<2'}
missing = [p for module, p in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
assert 1 <= EPOCHS <= 20 and BATCH_SIZE >= 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix
if tuple(int(v) for v in tf.__version__.split('.')[:2]) < (2, 16):
    raise RuntimeError('Use TensorFlow 2.16+; install requirements-notebooks.txt and restart the kernel.')
keras.utils.set_random_seed(MODEL_SEED)
tf.config.experimental.enable_op_determinism()
OUT = Path('outputs')/'ch17'/ARCH/str(uuid4())
OUT.mkdir(parents=True, exist_ok=False)
config = dict(architecture=ARCH, model_seed=MODEL_SEED, split_seed=SPLIT_SEED,
              epochs=EPOCHS, batch_size=BATCH_SIZE, use_gpu=USE_GPU)
software = dict(python=platform.python_version(), tensorflow=tf.__version__,
                numpy=np.__version__, pandas=pd.__version__)
print('Evidence directory:', OUT)
print('Devices:', tf.config.list_physical_devices())
print('Versions:', software)


In [ ]:
if importlib.util.find_spec('mne') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mne>=1.9,<2'])
import mne
if tuple(int(v) for v in mne.__version__.split('.')[:2]) < (1, 9):
    raise RuntimeError('Use MNE 1.9+ and restart the notebook kernel after installation.')
software['mne'] = mne.__version__


## 3. Load and inspect the task
[PhysioNet release](https://physionet.org/content/eegmmidb/1.0.0/) and [MNE run mapping](https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html): runs 4/8/12 are left/right hand **imagery**, T1 left and T2 right. The release states Open Data Commons Attribution License v1.0; retain attribution to Schalk and the PhysioNet release (DOI 10.13026/C28G6P).

This teaching pipeline chooses three channels, fixed 1–40 Hz filtering, the first two seconds after each task cue, and 64 Hz resampling after epoch extraction. The original recordings are 160 Hz. These are declared analysis choices, not a clinically optimized decoder. MNE applies anti-alias filtering during resampling; zeros or interpolation are not a substitute for that filter.

In [ ]:
from mne.datasets import eegbci
all_x, all_y, all_people, provenance = [], [], [], []
for person in SUBJECTS:
    files = eegbci.load_data(subjects=person, runs=RUNS, path='public-data', update_path=False, verbose=False)
    for run, filename in zip(RUNS, files):
        raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)
        raw.rename_channels(lambda name: name.rstrip('.'))
        raw.pick(CHANNELS)
        sfreq = float(raw.info['sfreq'])
        assert sfreq == 160.0, f'Unexpected sampling rate: {sfreq}'
        raw.filter(1., 40., verbose=False)
        events, _ = mne.events_from_annotations(raw, event_id={'T1': 1, 'T2': 2}, verbose=False)
        epochs = mne.Epochs(raw, events, event_id={'left': 1, 'right': 2},
                            tmin=0., tmax=2.-1./sfreq, baseline=None, preload=True, verbose=False)
        epochs.resample(64., verbose=False)
        # MNE: windows × channels × samples; Keras Conv1D: windows × samples × channels.
        values = epochs.get_data().transpose(0, 2, 1).astype('float32')
        assert values.shape[1:] == (128, 3)
        all_x.append(values); all_y.append(epochs.events[:, -1]-1)
        all_people.extend([person]*len(values))
        source_digest = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
        for event in epochs.events:
            provenance.append(dict(participant=person, run=run, event_sample=int(event[0]),
                                   target=int(event[-1]-1), original_sfreq=sfreq,
                                   source_file=Path(filename).name,
                                   source_sha256=source_digest))
X = np.concatenate(all_x); y = np.concatenate(all_y).astype('int64')
people = np.array(all_people)
assert len(provenance) == len(X) == len(y) == len(people)
pd.DataFrame(provenance).to_csv(OUT/'epoch_manifest.csv', index=False)
train = np.flatnonzero(np.isin(people, list(TRAIN_PEOPLE)))
validation = np.flatnonzero(np.isin(people, list(VAL_PEOPLE)))
test = np.flatnonzero(np.isin(people, list(TEST_PEOPLE)))
for indices in (train, validation, test):
    assert len(indices) and set(np.unique(y[indices])) == {0, 1}
print(pd.crosstab(people, y, rownames=['participant'], colnames=['target: 0 left / 1 right']))


## 4. Normalize using training people only
Filtering and epoching are fixed per-recording procedures. The channel mean and SD are learned across training epochs and must exclude validation/test people.

In [ ]:
mean = X[train].mean(axis=(0, 1), keepdims=True)
sd = X[train].std(axis=(0, 1), keepdims=True)
sd = np.where(sd > 0, sd, 1.)
X_train, X_val, X_test = [(X[i]-mean)/sd for i in (train, validation, test)]
y_train, y_val, y_test = y[train], y[validation], y[test]
np.savez(OUT/'normalization.npz', mean=mean, sd=sd)
split = dict(train_epoch_ids=train.tolist(), validation_epoch_ids=validation.tolist(),
             test_epoch_ids=test.tolist(), train_people=sorted(TRAIN_PEOPLE),
             validation_people=sorted(VAL_PEOPLE), test_people=sorted(TEST_PEOPLE))
(OUT/'split_manifest.json').write_text(json.dumps(split, indent=2))
config.update(dataset='PhysioNet EEGMMIDB 1.0.0', subjects=SUBJECTS, runs=RUNS,
              channels=CHANNELS, filter_hz=[1.,40.], target_sampling_hz=64,
              epoch_seconds=2., label_mapping={'0':'left-hand imagery','1':'right-hand imagery'},
              prediction_context='retrospective cue-aligned two-second epoch; not causal streaming')
plt.plot(np.arange(128)/64, X_val[0]); plt.xlabel('Seconds'); plt.ylabel('Training-standardized amplitude')
plt.legend(CHANNELS); plt.title('One validation epoch'); plt.show()


## 5. A compact temporal CNN
The kernels slide over samples. Channels retain fixed order; no rectangular scalp geometry is asserted. Two logits represent motor-imagery task labels.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(128, 3)),
    layers.Conv1D(8, 9, padding='same', activation='relu'),
    layers.AveragePooling1D(2),
    layers.Conv1D(16, 5, padding='same', activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(2)
])
model.summary()


## 6. Train and select on validation people


In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
                    callbacks=[keras.callbacks.EarlyStopping(
                        monitor='val_loss', patience=2, restore_best_weights=True)])
selected_epoch = int(np.argmin(history.history['val_loss'])) + 1
pd.DataFrame(history.history).to_csv(OUT/'history.csv', index=False)
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='validation')
plt.xlabel('Training epoch (zero-based plot index)'); plt.ylabel('Cross-entropy')
plt.legend(); plt.tight_layout(); plt.savefig(OUT/'learning_curves.png', dpi=160); plt.show()
print('Selected epoch:', selected_epoch, '| Executed epochs:', len(history.history['loss']))


## 7. Evaluate the held-out task epochs
People have both task labels, so keep each epoch target. Two held-out people give very limited evidence; epoch counts are not independent participant counts.

In [ ]:
probabilities = tf.nn.softmax(model.predict(X_test, verbose=0), axis=-1).numpy()
predictions = probabilities.argmax(axis=1)
table = pd.DataFrame({'epoch_id':test, 'participant':people[test], 'target':y_test,
                      'probability_right':probabilities[:, 1], 'prediction':predictions})
table['correct'] = table.target == table.prediction
per_person = table.groupby('participant')['correct'].mean()
print('Per-person epoch accuracy:', per_person.to_dict())
print('Macro mean of test-person accuracies:', float(per_person.mean()))
print('Epoch confusion matrix, rows observed / columns predicted:', confusion_matrix(y_test, predictions, labels=[0,1]))
table.to_csv(OUT/'predictions.csv', index=False)


### Compare a small classical model on the same people
Use fixed 8–13 and 13–30 Hz Fourier summaries, three channels, and training-only feature scaling. These are simple finite-window power summaries, not a calibrated PSD estimate or an optimized BCI baseline.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
frequencies = np.fft.rfftfreq(128, 1/64)
fourier_power = np.abs(np.fft.rfft(X, axis=1))**2 / 128**2
features = np.concatenate([np.log(np.maximum(
    fourier_power[:, (frequencies>=lo)&(frequencies<hi)].mean(axis=1), 1e-30))
    for lo, hi in [(8,13),(13,30)]], axis=1)
baseline = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=1000))
baseline.fit(features[train], y[train])
baseline_predictions = baseline.predict(features[test])
baseline_table = pd.DataFrame({'participant':people[test], 'correct':baseline_predictions==y_test})
baseline_macro = float(baseline_table.groupby('participant').correct.mean().mean())
print('Classical macro test-person accuracy:', baseline_macro)
report = dict(task='left/right motor imagery', selected_epoch=selected_epoch,
              test_people=sorted(TEST_PEOPLE), test_epochs=len(test),
              per_person_epoch_accuracy={str(k):float(v) for k,v in per_person.items()},
              macro_person_accuracy=float(per_person.mean()), classical_macro_accuracy=baseline_macro,
              epoch_brier=float(np.mean((probabilities[:,1]-y_test)**2)),
              parameters=int(model.count_params()), clinical_validation=False)


## 8. Keep the schema, checkpoint, normalization and provenance together


In [ ]:
model.save_weights(OUT/'selected.weights.h5')
(OUT/'architecture.json').write_text(model.to_json())
(OUT/'config.json').write_text(json.dumps(config, indent=2))
(OUT/'model_card.md').write_text(
    '# Teaching model\n\nPublic data, educational use. No clinical validation. '
    'Validation loss selected the checkpoint; test data did not. '
    'Weights alone are not a complete training-resume checkpoint. '
    'Architecture is supplied in this notebook; retain the notebook and environment.\n')
report.update(software=software, config=config, review_status='review_required', accepted=False)
report['artifact_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                            for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'report.json').write_text(json.dumps(report, indent=2))
print('Saved:', sorted(p.name for p in OUT.iterdir()))
print('Download this directory before ending an ephemeral Colab session; keep it outside Git.')


## Explain before celebrating
1. Why would a random epoch split answer a different question?
2. Why is averaging every prediction into one participant diagnosis wrong for this task?
3. Did the CNN improve over the fixed classical baseline on these two people? What remains uncertain either way?
4. Fixed zero-phase filtering and a complete cue-aligned epoch support retrospective decoding, not instantaneous causal streaming. What would online use require?
5. What would change with additional participants, different sessions, more channels or another device?

No favorable accuracy is guaranteed. Keep the original result, inspect labels and shapes, and discuss uncertainty instead of searching the test set for a pleasing score. The original synthetic PyTorch lab remains available as the advanced engineering track.